# Practical 3: Convolution Arithmetic: Stride, Padding & Pooling
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit II (Convolutional Neural Networks) | **CO:** CO2  
**Problem Statement Context:** Spatial Downsampling & Receptive Field Expansion in Foliar Vision  

---

## 1. Feature Map Spatial Arithmetic
Given an input feature map of size $W_{in} 	imes H_{in}$, kernel size $K$, padding $P$, and stride $S$, the output spatial dimensions are:

$$W_{out} = \left\lfloor rac{W_{in} - K + 2P}{S} ightfloor + 1$$
$$H_{out} = \left\lfloor rac{H_{in} - K + 2P}{S} ightfloor + 1$$

### Padding Modes:
1. **Valid Padding ($P=0$):** No boundary zero-padding; output feature map shrinks with each layer.
2. **Same Padding ($P = \lfloor K/2 floor$):** Padding selected such that $W_{out} = W_{in}$ when $S=1$.

### Pooling Operations:
- **Max Pooling:** Selects the maximum activation in a window $\Omega$, preserving dominant edge/texture activations while providing translation invariance:
  $$y_{i, j} = \max_{(m, n) \in \Omega} x_{i \cdot s + m, j \cdot s + n}$$
- **Average Pooling:** Computes arithmetic mean, smoothing background features.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from notebooks import make_nb02  # reference

## 2. Pooling Functions from Scratch

In [ ]:
def max_pool2d_scratch(image: np.ndarray, pool_size: int = 2, stride: int = 2) -> np.ndarray:
    """Max pooling from first principles."""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)

    for i in range(out_h):
        for j in range(out_w):
            patch = image[i*stride : i*stride + pool_size, j*stride : j*stride + pool_size]
            output[i, j] = np.max(patch)

    return output


def avg_pool2d_scratch(image: np.ndarray, pool_size: int = 2, stride: int = 2) -> np.ndarray:
    """Average pooling from first principles."""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)

    for i in range(out_h):
        for j in range(out_w):
            patch = image[i*stride : i*stride + pool_size, j*stride : j*stride + pool_size]
            output[i, j] = np.mean(patch)

    return output

print("[✓] Pooling functions defined.")

## 3. Empirical Demonstration on Leaf Patch

In [ ]:
sample_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Early_Blight" / "sample_000.jpg"
leaf_gray = np.array(Image.open(sample_path).convert("L"), dtype=np.float32) / 255.0

# Sub-patch around lesion
patch = leaf_gray[60:188, 60:188]  # 128x128 patch

max_pooled = max_pool2d_scratch(patch, pool_size=2, stride=2)   # 64x64
avg_pooled = avg_pool2d_scratch(patch, pool_size=2, stride=2)   # 64x64

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(patch, cmap="gray")
axes[0].set_title(f"Input Crop Patch ({patch.shape[0]}x{patch.shape[1]})")
axes[0].axis("off")

axes[1].imshow(max_pooled, cmap="gray")
axes[1].set_title(f"Max Pooling 2x2 ({max_pooled.shape[0]}x{max_pooled.shape[1]}) - High Contrast")
axes[1].axis("off")

axes[2].imshow(avg_pooled, cmap="gray")
axes[2].set_title(f"Avg Pooling 2x2 ({avg_pooled.shape[0]}x{avg_pooled.shape[1]}) - Smoothed")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 4. Receptive Field Analysis
The receptive field (RF) denotes the region in the input space that a particular CNN feature is looking at. For a sequence of layers:
$$RF_{l} = RF_{l-1} + (k_l - 1) 	imes j_{l-1}$$
where $j_{l} = j_{l-1} 	imes s_l$ is the cumulative stride.

Pooling operations double the cumulative jump ($j$), accelerating receptive field expansion so deep layers can encompass entire crop leaves rather than localized pixel spots.